In [2]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
import faiss
import ollama
import joblib
import shap

import time
import os
import psutil

In [3]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index("../data/processed/knowledge_base2_faiss.index")
knowledge_base = pd.read_csv("../data/processed/knowledge_base_chunks.csv")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
print(index.ntotal)
print(knowledge_base.shape)

137
(137, 4)


In [17]:
def retrieve_context(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    query_embedding = normalize(
        query_embedding,
        norm="l2"
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    retrieved_docs = knowledge_base.iloc[indices[0]].copy()
    retrieved_docs["similarity_score"] = distances[0]

    return retrieved_docs.reset_index(drop=True)

In [18]:
query = (
    "My product stopped working after installation. "
    "I already tried a factory reset, but the problem continues. "
    "What should I do?"
)

retrieved_docs = retrieve_context(query, top_k=5)

print("Customer Query:")
print(query)

print("\n")

for rank, row in retrieved_docs.iterrows():
    print(f"--- Document {rank + 1} ---")
    print(f"Similarity: {row['similarity_score']:.4f}")
    print(f"Source: {row['source']}")
    print(f"Document ID: {row['document_id']}")
    print(f"Chunk ID: {row['chunk_id']}")
    print(f"Text: {row['text']}")
    print()

Customer Query:
My product stopped working after installation. I already tried a factory reset, but the problem continues. What should I do?


--- Document 1 ---
Similarity: 0.5917
Source: technical_support.txt
Document ID: technical_support
Chunk ID: 3
Text: If a product does not work after installation, the customer should confirm that the installation was completed correctly and describe what happens when the product is started.

--- Document 2 ---
Similarity: 0.5682
Source: technical_support.txt
Document ID: technical_support
Chunk ID: 4
Text: If the customer has already performed a factory reset and the problem continues, the issue should be escalated for further technical investigation rather than repeatedly recommending the same reset procedure.

--- Document 3 ---
Similarity: 0.5524
Source: technical_support.txt
Document ID: technical_support
Chunk ID: 5
Text: If a product stopped working after a software update, the issue should be reviewed to determine whether the update may 

In [26]:
context = "\n\n".join(
    [
        f"Support Document {i + 1}:\n{row['text']}"
        for i, (_, row) in enumerate(retrieved_docs.iterrows())
    ]
)

print("RAG context created successfully.")
print("\nContext:\n")
print(context)

RAG context created successfully.

Context:

Support Document 1:
If a product does not work after installation, the customer should confirm that the installation was completed correctly and describe what happens when the product is started.

Support Document 2:
If the customer has already performed a factory reset and the problem continues, the issue should be escalated for further technical investigation rather than repeatedly recommending the same reset procedure.

Support Document 3:
If a product stopped working after a software update, the issue should be reviewed to determine whether the update may be related to the problem.

Support Document 4:
If the same problem continues after documented troubleshooting steps have been completed, the issue should be escalated for further investigation.

Support Document 5:
If a product is defective during the applicable warranty period, the case should be reviewed by customer support.


In [27]:
rag_prompt = f"""
You are a customer support assistant.

Answer the customer's question using only the information
provided in the support context below.

If the context does not contain enough information,
clearly say that more information or human support is needed.

Do not invent product-specific facts or solutions.

Customer Question:
{query}

Support Context:
{context}


"""

print(rag_prompt)


You are a customer support assistant.

Answer the customer's question using only the information
provided in the support context below.

If the context does not contain enough information,
clearly say that more information or human support is needed.

Do not invent product-specific facts or solutions.

Customer Question:
My product stopped working after installation. I already tried a factory reset, but the problem continues. What should I do?

Support Context:
Support Document 1:
If a product does not work after installation, the customer should confirm that the installation was completed correctly and describe what happens when the product is started.

Support Document 2:
If the customer has already performed a factory reset and the problem continues, the issue should be escalated for further technical investigation rather than repeatedly recommending the same reset procedure.

Support Document 3:
If a product stopped working after a software update, the issue should be reviewed to 

In [28]:
response = ollama.chat(
    model="llama3.2:3b",
    messages=[{"role": "user", "content": rag_prompt}]
)

answer = response["message"]["content"]

print("RAG Answer:")
print(answer)

RAG Answer:
Based on the information provided, I would like to ask a few clarifying questions to help resolve the issue.

Can you please confirm that the installation was completed correctly and describe what happens when the product is started? Additionally, you've already tried a factory reset, but the problem persists. According to our Support Document 1, we should investigate further to determine the cause of the issue.

Would you like to proceed with escalating the issue for further technical investigation or would you like to explore other troubleshooting steps?


In [29]:
test_queries = [
    "My product is not working after installation.",
    "I want to cancel my order. What should I do?",
    "I was charged incorrectly on my bill.",
    "I need help with a technical problem.",
    "I want to request a refund."
]

for query in test_queries:
    print("=" * 80)
    print("Customer Query:", query)

    retrieved_docs = retrieve_context(query, top_k=3)
    context = "\n\n".join(retrieved_docs["text"].tolist())

    rag_prompt = f"""
You are a customer support assistant.

Answer the customer's question using only the information
provided in the support context below.

If the context does not contain enough information,
clearly say that more information or human support is needed.

Do not invent product-specific facts or solutions.

Customer Question:
{query}

Support Context:
{context}

Answer:
"""

    response = ollama.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": rag_prompt}]
    )

    print("RAG Answer:")
    print(response["message"]["content"])
    print()

Customer Query: My product is not working after installation.
RAG Answer:
To help resolve the issue with your product, can you please confirm that the installation was completed correctly and describe what happens when you start the product? Additionally, if a software update was installed recently, please let me know if that may be related to the problem.

Customer Query: I want to cancel my order. What should I do?
RAG Answer:
I'd be happy to help you with cancelling your order. However, I need a bit more information from you. Could you please provide your order number so I can check on the current status of your order and see what options are available for cancellation?

Customer Query: I was charged incorrectly on my bill.
RAG Answer:
To resolve the billing issue, I recommend that you contact our billing support team. They will review your billing records and investigate the incorrect charge.

Customer Query: I need help with a technical problem.
RAG Answer:
I'd be happy to help yo

In [33]:
print("RAG PIPELINE TEST SUMMARY")

print("Model       : Llama 3.2 3B")
print("Embeddings  : all-MiniLM-L6-v2")
print("Vector DB   : FAISS")
print("Knowledge Base: Business Knowledge Base")
print("Queries Tested:", len(test_queries))
print("Status      : SUCCESS")

RAG PIPELINE TEST SUMMARY
Model       : Llama 3.2 3B
Embeddings  : all-MiniLM-L6-v2
Vector DB   : FAISS
Knowledge Base: Business Knowledge Base
Queries Tested: 25
Status      : SUCCESS


In [31]:
test_queries = [
    "My product stopped working after installation. What should I do?",
    "I am having a technical problem with my product.",
    "My device is not turning on even after charging it.",
    "The product stopped working after a recent software update.",
    "I cannot complete the product setup. Can you help me?",
    "I want to cancel my order. What is the process?",
    "Please help me cancel my recent purchase.",
    "I received the wrong product and want to return it.",
    "I would like to request a refund for my purchase.",
    "I have not received my order yet. What should I do?",
    "My order has been delayed and I need an update.",
    "I was charged the wrong amount on my bill.",
    "There is an unexpected charge on my account.",
    "I was charged twice for the same purchase.",
    "Can you explain why I was charged an additional amount?",
    "I need help with a billing issue.",
    "The payment for my order failed. What should I do?",
    "My payment was successful but the order was not confirmed.",
    "I cannot access my account after changing my password.",
    "I am having trouble logging into my account.",
    "My product arrived damaged. How can I get a replacement?",
    "The product I received is not working correctly.",
    "I need help with a product warranty issue.",
    "How can I contact customer support about my problem?",
    "I have tried the suggested solution but my issue is still not resolved."
]

In [32]:
rag_test_queries = pd.DataFrame({
    "query_id": range(1, len(test_queries) + 1),
    "customer_query": test_queries
})

rag_test_queries.to_csv(
    "../data/processed/rag_test_queries.csv",
    index=False
)

print(f"Saved")
print(rag_test_queries.head())

Saved
   query_id                                     customer_query
0         1  My product stopped working after installation....
1         2   I am having a technical problem with my product.
2         3  My device is not turning on even after chargin...
3         4  The product stopped working after a recent sof...
4         5  I cannot complete the product setup. Can you h...


### Prompt Engineering

In [40]:
prompt_template = """
You are an AI customer support assistant.

Your task is to answer the customer's question using the
provided customer-support context.

Rules:
1. Use only the information available in the context.
2. Do not invent product-specific facts, policies, or solutions.
3. If the context is insufficient, clearly state that more
   information or human support is required.
4. Give a concise and helpful response.
5. Do not mention the retrieval process, embeddings, or FAISS.
6. Maintain a professional and polite customer-support tone.

Customer Question:
{query}

Support Context:
{context}

Answer:
"""

In [35]:
query = "My product is not working after installation."

retrieved_docs = retrieve_context(query, top_k=3)

context = "\n\n".join(retrieved_docs["text"].tolist())

prompt = prompt_template.format(query=query, context=context)
response = ollama.chat(
    model="llama3.2:3b",
    messages=[{"role": "user", "content": prompt}]
)

print("Customer Query:")
print(query)

print("\nAI Response:")
print(response["message"]["content"])

Customer Query:
My product is not working after installation.

AI Response:
I'm sorry to hear that your product is not working after installation. Can you please confirm that the installation was completed correctly? Additionally, could you describe what happens when you start the product? This will help me better understand the issue and assist you further.


In [36]:
improved_prompt_template = """
You are a professional customer support assistant.

Answer the customer's question using ONLY the information
provided in the support context.

Rules:

- Give a direct, concise, and helpful answer.
- Use only information explicitly supported by the context.
- Do not invent facts, troubleshooting steps, policies,
  product details, or customer-specific information.
- Do not assume that an action has already been completed
  unless the customer explicitly states it.
- If the support information explicitly recommends escalation,
  clearly recommend contacting human customer support.
- Do not guarantee refunds, cancellations, replacements,
  warranty approval, payment resolution, or delivery dates
  unless the context explicitly supports the statement.
- If the available information is insufficient to provide
  a specific solution, say:
  "The available support information is insufficient to provide
  a specific solution. Human support may be required."
- Do not ask for unnecessary personal or sensitive information.
- Do not mention FAISS, embeddings, retrieval, vector databases,
  the context, or the language model.
- Maintain a professional and polite customer-support tone.
- Keep the answer within 3-5 sentences.

Customer Question:
{query}

Support Information:
{context}

Answer:
"""

In [37]:
prompt = improved_prompt_template.format(
    query=query,
    context=context
)

response = ollama.chat(
    model="llama3.2:3b",
    messages=[{"role": "user", "content": prompt}]
)

print("Improved AI Response:")
print(response["message"]["content"])

Improved AI Response:
I'm sorry to hear that your product is not working after installation. Can you please confirm that the installation was completed correctly and describe what happens when the product is started?


In [38]:
prompt_comparison = pd.DataFrame({
    "Version": [
        "Original Prompt",
        "Improved Prompt"
    ],
    "Approach": [
        "General RAG instructions",
        "Strict context-only instructions"
    ],
    "Response_Control": [
        "Moderate",
        "Strong"
    ],
    "Hallucination_Control": [
        "Basic",
        "High"
    ],
    "Escalation_Handling": [
        "General",
        "Explicit"
    ]
})

prompt_comparison

,Version,Approach,Response_Control,Hallucination_Control,Escalation_Handling
0,Original Prompt,General RAG instructions,Moderate,Basic,General
1,Improved Prompt,Strict context-only instructions,Strong,High,Explicit


In [39]:
prompt_comparison.to_csv("../data/processed/prompt_engineering_comparison.csv", index=False)

print("saved successfully.")

saved successfully.


### Tool Calling

In [44]:
customer_data = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

customer_data["TotalCharges"] = pd.to_numeric(customer_data["TotalCharges"], errors="coerce")
print(customer_data.shape)

(7043, 21)


In [45]:
# creating customer profile tool

def get_customer_profile(customer_id):
    result = customer_data[
        customer_data["customerID"].astype(str) == str(customer_id)
    ]

    if result.empty:
        return {
            "status": "not_found",
            "message": "Customer not found."
        }
    customer = result.iloc[0]

    return {
        "status": "success",
        "customer_id": customer["customerID"],
        "gender": customer["gender"],
        "tenure": customer["tenure"],
        "contract": customer["Contract"],
        "monthly_charges": customer["MonthlyCharges"],
        "internet_service": customer["InternetService"],
        "payment_method": customer["PaymentMethod"],
        "churn": customer["Churn"]
    }

print(get_customer_profile("7590-VHVEG"))

{'status': 'success', 'customer_id': '7590-VHVEG', 'gender': 'Female', 'tenure': np.int64(1), 'contract': 'Month-to-month', 'monthly_charges': np.float64(29.85), 'internet_service': 'DSL', 'payment_method': 'Electronic check', 'churn': 'No'}


In [46]:
selected_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "IsFiberOptic",
    "NoInternet",
    "ChargeToTenureRatio",
    "HighRiskCustomer",
    "InternetService_Fiber optic",
    "InternetService_No",
    "OnlineSecurity_No",
    "OnlineSecurity_No internet service",
    "OnlineBackup_No",
    "OnlineBackup_No internet service",
    "DeviceProtection_No",
    "DeviceProtection_No internet service",
    "TechSupport_No",
    "TechSupport_No internet service",
    "StreamingTV_No internet service",
    "StreamingMovies_No internet service",
    "Contract_Month-to-month",
    "Contract_Two year",
    "PaperlessBilling_No",
    "PaperlessBilling_Yes",
    "PaymentMethod_Electronic check",
    "TenureGroup_New Customer",
    "TenureGroup_Very Long Term",
    "MonthlyChargesGroup_High",
    "MonthlyChargesGroup_Low",
    "InternetTechSupport_Internet_Without_Support",
    "InternetTechSupport_No_Internet"
]

In [47]:
def prepare_customer_features(customer):

    df = pd.DataFrame([customer])

    # Convert TotalCharges to numeric
    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"],
        errors="coerce"
    )

    df["TotalCharges"] = df["TotalCharges"].fillna(
        customer_data["TotalCharges"].median()
    )

    # Tenure Group
    df["TenureGroup"] = pd.cut(
        df["tenure"],
        bins=[-1, 12, 24, 48, 72],
        labels=[
            "New Customer",
            "Short Term",
            "Medium Term",
            "Long Term"
        ]
    )

    # Monthly Charges Group
    df["MonthlyChargesGroup"] = pd.cut(
        df["MonthlyCharges"],
        bins=[-1, 40, 70, float("inf")],
        labels=[
            "Low",
            "Medium",
            "High"
        ]
    )

    # Total Charges Group
    df["TotalChargesGroup"] = pd.cut(
        df["TotalCharges"],
        bins=[-1, 1000, 3000, float("inf")],
        labels=[
            "Low",
            "Medium",
            "High"
        ]
    )

    # Number of Services
    service_columns = [
        "PhoneService",
        "MultipleLines",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies"
    ]

    df["NumberOfServices"] = 0

    for column in service_columns:
        if column in df.columns:
            df["NumberOfServices"] += (
                df[column].astype(str).isin(["Yes", "Yes"]).astype(int)
            )

    # Internet Service Features
    df["IsDSL"] = (
        df["InternetService"] == "DSL"
    ).astype(int)

    df["IsFiberOptic"] = (
        df["InternetService"] == "Fiber optic"
    ).astype(int)

    df["NoInternet"] = (
        df["InternetService"] == "No"
    ).astype(int)

    # Charge to Tenure Ratio
    df["ChargeToTenureRatio"] = 0.0

    valid_tenure = df["tenure"] > 0

    df.loc[valid_tenure, "ChargeToTenureRatio"] = (
        df.loc[valid_tenure, "TotalCharges"]
        / df.loc[valid_tenure, "tenure"]
    )

    # High Value Customer
    df["HighValueCustomer"] = (
        (df["MonthlyCharges"] >= 70) &
        (df["TotalCharges"] >= 3000)
    ).astype(int)

    # High Risk Customer
    df["HighRiskCustomer"] = (
        (df["tenure"] <= 12) &
        (df["Contract"] == "Month-to-month") &
        (df["MonthlyCharges"] >= 70)
    ).astype(int)

    # Internet + Technical Support
    df["InternetTechSupport"] = "No_Internet"

    df.loc[
        (df["InternetService"] != "No") &
        (df["TechSupport"] == "Yes"),
        "InternetTechSupport"
    ] = "Internet_With_Support"

    df.loc[
        (df["InternetService"] != "No") &
        (df["TechSupport"] != "Yes"),
        "InternetTechSupport"
    ] = "Internet_Without_Support"

    # Categorical encoding
    categorical_columns = [
        "gender",
        "SeniorCitizen",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod",
        "TenureGroup",
        "MonthlyChargesGroup",
        "TotalChargesGroup",
        "InternetTechSupport"
    ]

    df = pd.get_dummies(
        df,
        columns=categorical_columns,
        dtype=int
    )

    # Make sure all selected features exist
    for feature in selected_features:
        if feature not in df.columns:
            df[feature] = 0

    # Keep exactly the 30 selected model features
    return df[selected_features].copy()

In [51]:
# SHAP Explanation Tool

explainer = shap.TreeExplainer(model)

def explain_customer_churn(customer_id):

    result = customer_data[
        customer_data["customerID"].astype(str) == str(customer_id)
    ]

    if result.empty:
        return {
            "status": "not_found",
            "message": "Customer not found."
        }

    customer = result.iloc[0].to_dict()

    X_customer = prepare_customer_features(customer)

    shap_values = explainer.shap_values(X_customer)

    if isinstance(shap_values, list):
        values = shap_values[1][0]
    else:
        values = shap_values[0]

    explanation = pd.DataFrame({
        "feature": X_customer.columns,
        "shap_value": values
    })

    explanation["impact"] = explanation["shap_value"].apply(
        lambda x:
        "Increases churn risk"
        if x > 0
        else "Decreases churn risk"
    )

    explanation["abs_shap"] = explanation["shap_value"].abs()

    explanation = (
        explanation
        .sort_values("abs_shap", ascending=False)
        .head(5)
    )

    return {
        "status": "success",
        "customer_id": customer_id,
        "top_factors": [
            {
                "feature": row["feature"],
                "shap_value": round(float(row["shap_value"]), 4),
                "impact": row["impact"]
            }
            for _, row in explanation.iterrows()
        ]
    }

print(explain_customer_churn("7590-VHVEG"))

{'status': 'success', 'customer_id': '7590-VHVEG', 'top_factors': [{'feature': 'tenure', 'shap_value': -0.6394, 'impact': 'Decreases churn risk'}, {'feature': 'ChargeToTenureRatio', 'shap_value': 0.4305, 'impact': 'Increases churn risk'}, {'feature': 'OnlineSecurity_No', 'shap_value': -0.2935, 'impact': 'Decreases churn risk'}, {'feature': 'Contract_Month-to-month', 'shap_value': 0.2901, 'impact': 'Increases churn risk'}, {'feature': 'Contract_Two year', 'shap_value': 0.2857, 'impact': 'Increases churn risk'}]}


In [52]:
# Knowledge Search Tool

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

index = faiss.read_index("../data/processed/knowledge_base2_faiss.index")
knowledge_base = pd.read_csv("../data/processed/knowledge_base_chunks.csv")

def search_customer_knowledge(query, top_k=5):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)

    query_embedding = normalize(query_embedding, norm="l2").astype("float32")

    distances, indices = index.search(query_embedding,top_k)

    results = knowledge_base.iloc[indices[0]].copy()
    results["similarity_score"] = distances[0]

    return {
        "status": "success",
        "query": query,
        "results": results[
            [
                "document_id",
                "source",
                "chunk_id",
                "text",
                "similarity_score"
            ]
            ].to_dict("records")
    }

print(search_customer_knowledge("My product stopped working after installation."))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'status': 'success', 'query': 'My product stopped working after installation.', 'results': [{'document_id': 'technical_support', 'source': 'technical_support.txt', 'chunk_id': 3, 'text': 'If a product does not work after installation, the customer should confirm that the installation was completed correctly and describe what happens when the product is started.', 'similarity_score': 0.6411415934562683}, {'document_id': 'technical_support', 'source': 'technical_support.txt', 'chunk_id': 5, 'text': 'If a product stopped working after a software update, the issue should be reviewed to determine whether the update may be related to the problem.', 'similarity_score': 0.5105657577514648}, {'document_id': 'technical_support', 'source': 'technical_support.txt', 'chunk_id': 10, 'text': 'If the same problem continues after documented troubleshooting steps have been completed, the issue should be escalated for further investigation.', 'similarity_score': 0.40628546476364136}, {'document_id': 'te

In [53]:
# rag answer tool

def generate_rag_answer(query, top_k=5):
    search_result = search_customer_knowledge(query, top_k=top_k)
    context = "\n\n".join([item["text"] for item in search_result["results"]])

    prompt = improved_prompt_template.format(query=query, context=context)
    response = ollama.chat(model="llama3.2:3b", messages=[{"role": "user", "content": prompt}])

    return {
        "status": "success",
        "query": query,
        "answer": response["message"]["content"],
        "sources": search_result["results"]
    }


rag_result = generate_rag_answer("My product stopped working after installation.")

print("Answer:")
print(rag_result["answer"])

print("\nSources:")

for source in rag_result["sources"]:
    print(
        source["source"],
        "| chunk:",
        source["chunk_id"],
        "| similarity:",
        round(float(source["similarity_score"]), 4)
    )

Answer:
I'd be happy to help you with your product issue. Can you please confirm that the installation was completed correctly and describe what happens when you start the product?

Sources:
technical_support.txt | chunk: 3 | similarity: 0.6411
technical_support.txt | chunk: 5 | similarity: 0.5106
technical_support.txt | chunk: 10 | similarity: 0.4063
technical_support.txt | chunk: 4 | similarity: 0.3728
warranty_policy.txt | chunk: 3 | similarity: 0.3695


##### 
The RAG pipeline was evaluated using customer-support queries by checking retrieval relevance, contextual grounding, hallucination avoidance, and response appropriateness. When the retrieved knowledge did not contain sufficient information, the system correctly avoided generating unsupported solutions and indicated that human support may be required

In [55]:
# Combined Tool Test
customer_id = "7590-VHVEG"
query = "My product stopped working after installation."

print("\n1. CUSTOMER PROFILE")
profile = get_customer_profile(customer_id)
print(profile)

print("\n2. CHURN PREDICTION")
churn_result = predict_customer_churn(customer_id)
print(churn_result)

print("\n3. SHAP EXPLANATION")
shap_result = explain_customer_churn(customer_id)
print(shap_result)

print("\n4. KNOWLEDGE SEARCH")
knowledge_result = search_customer_knowledge(query, top_k=3)
print(knowledge_result)

print("\n5. RAG ANSWER")
rag_result = generate_rag_answer(query, top_k=3)
print(rag_result["answer"])

print("\n")
print("executed successfullly")


1. CUSTOMER PROFILE
{'status': 'success', 'customer_id': '7590-VHVEG', 'gender': 'Female', 'tenure': np.int64(1), 'contract': 'Month-to-month', 'monthly_charges': np.float64(29.85), 'internet_service': 'DSL', 'payment_method': 'Electronic check', 'churn': 'No'}

2. CHURN PREDICTION
{'status': 'success', 'customer_id': '7590-VHVEG', 'churn_prediction': 'No Churn', 'churn_probability': np.float32(37.8), 'threshold': 0.6}

3. SHAP EXPLANATION
{'status': 'success', 'customer_id': '7590-VHVEG', 'top_factors': [{'feature': 'tenure', 'shap_value': -0.6394, 'impact': 'Decreases churn risk'}, {'feature': 'ChargeToTenureRatio', 'shap_value': 0.4305, 'impact': 'Increases churn risk'}, {'feature': 'OnlineSecurity_No', 'shap_value': -0.2935, 'impact': 'Decreases churn risk'}, {'feature': 'Contract_Month-to-month', 'shap_value': 0.2901, 'impact': 'Increases churn risk'}, {'feature': 'Contract_Two year', 'shap_value': 0.2857, 'impact': 'Increases churn risk'}]}

4. KNOWLEDGE SEARCH
{'status': 'succe

In [56]:
# Simple Agent Workflow

def simple_agent(query, customer_id=None):

    query_lower = query.lower()

    # Support / RAG queries first
    support_keywords = [
        "support", "product", "refund", "billing",
        "technical", "installation", "order",
        "payment", "charged", "charge", "working",
        "broken", "failed", "delivery", "return",
        "replacement", "warranty"
    ]

    if any(word in query_lower for word in support_keywords):
        return {"tool": "RAG Answer", "result": generate_rag_answer(query)}

    # SHAP explanation
    if customer_id and any(word in query_lower for word in ["why", "reason", "explain"]):
        return {
            "tool": "SHAP Explanation",
            "result": explain_customer_churn(customer_id)
        }

    # Churn prediction
    if customer_id and any(word in query_lower for word in ["churn", "risk", "leave"]):
        return {
            "tool": "Churn Prediction",
            "result": predict_customer_churn(customer_id)
        }


    # Customer profile
    if customer_id:
        return {
            "tool": "Customer Profile",
            "result": get_customer_profile(customer_id)
        }

    # Default
    return {
        "tool": "Knowledge Search",
        "result": search_customer_knowledge(query)
    }

In [58]:
agent_tests = [
    ("What is my churn risk?", "7590-VHVEG"),
    ("Why is this customer at risk?", "7590-VHVEG"),
    ("My product stopped working after installation.", None),
    ("I need help with a billing issue.", None),
    ("Show me this customer's information.", "7590-VHVEG")
]

for query, customer_id in agent_tests:
    result = simple_agent(query, customer_id=customer_id)

    print("Query:", query)
    print("Selected Tool:", result["tool"])
    print("Status:", result["result"].get("status"))

Query: What is my churn risk?
Selected Tool: Churn Prediction
Status: success
Query: Why is this customer at risk?
Selected Tool: SHAP Explanation
Status: success
Query: My product stopped working after installation.
Selected Tool: RAG Answer
Status: success
Query: I need help with a billing issue.
Selected Tool: RAG Answer
Status: success
Query: Show me this customer's information.
Selected Tool: Customer Profile
Status: success


### Gen AI Evaluation

In [60]:
genai_test_queries = [
    "What is my churn risk?",
    "Why is this customer at risk?",
    "My product stopped working after installation.",
    "I need help with a billing issue.",
    "Show me this customer's information.",
    "I want to request a refund.",
    "My payment failed. What should I do?",
    "Can you explain my customer's churn prediction?",
    "I am having a technical problem with my product.",
    "Tell me about this customer."
]

print("GenAI evaluation queries:", len(genai_test_queries))

GenAI evaluation queries: 10


In [61]:
evaluation_results = []

for query in genai_test_queries:

    result = simple_agent(query, customer_id="7590-VHVEG")
    evaluation_results.append({
        "query": query,
        "selected_tool": result["tool"],
        "status": result["result"].get("status", "unknown")
    })

genai_evaluation = pd.DataFrame(evaluation_results)
genai_evaluation

,query,selected_tool,status
0,What is my churn risk?,Churn Prediction,success
1,Why is this customer at risk?,SHAP Explanation,success
2,My product stopped working after installation.,RAG Answer,success
3,I need help with a billing issue.,RAG Answer,success
4,Show me this customer's information.,Customer Profile,success
5,I want to request a refund.,RAG Answer,success
6,My payment failed. What should I do?,RAG Answer,success
7,Can you explain my customer's churn prediction?,SHAP Explanation,success
8,I am having a technical problem with my product.,RAG Answer,success
9,Tell me about this customer.,Customer Profile,success


In [62]:
genai_evaluation.to_csv("../data/processed/genai_agent_evaluation.csv", index=False)
print(len(genai_evaluation))

10


### Agent Evaluation: 
    The Simple Agent was evaluated using 10 representative customer queries covering churn prediction, SHAP-based explanation, customer profile retrieval, knowledge search, and RAG-based customer support. All 10 queries were correctly routed to their intended tools and completed successfully, resulting in 100% tool-routing accuracy for the evaluation set.

### Resource Benchmarking

In [63]:
process = psutil.Process(os.getpid())

In [87]:
customer_id = "7590-VHVEG"
query = "My product stopped working after installation."

In [88]:
# RAM before
ram_before = process.memory_info().rss / (1024 ** 2)

In [89]:
# model loading time

start = time.perf_counter()
test_model = joblib.load("../models/final_xgb_feature_model.pkl")
ml_time = time.perf_counter() - start

In [90]:
# ML inference time

X_customer = prepare_customer_features(
    customer_data[
        customer_data["customerID"] == customer_id
    ].iloc[0].to_dict()
)

start = time.perf_counter()
test_model.predict_proba(X_customer)
ml_inference_time = time.perf_counter() - start

In [91]:
#  Embedding generation time
start = time.perf_counter()
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
)
em_generation_time = time.perf_counter() - start

query_embedding = normalize(
    query_embedding,
    norm="l2"
).astype("float32")

In [92]:
# FAISS search time
start = time.perf_counter()
index.search(query_embedding, 5)
faiss_time = time.perf_counter() - start

In [93]:
# RAG retrieval time
start = time.perf_counter()
search_customer_knowledge(query, top_k=5)
rag_time = time.perf_counter() - start

In [94]:
# LLM inference time
search_result = search_customer_knowledge(query, top_k=5)

context = "\n\n".join(item["text"] for item in search_result["results"])
prompt = improved_prompt_template.format(query=query, context=context)

start = time.perf_counter()
ollama.chat(model="llama3.2:3b", messages=[{"role": "user", "content": prompt}])
llm_time = time.perf_counter() - start

In [95]:
# End-to-end response time
start = time.perf_counter()
simple_agent(query, customer_id)
end_to_end_time = time.perf_counter() - start

In [96]:
# RAM after
ram_after = process.memory_info().rss / (1024 ** 2)

In [100]:
print("System Resource Benchmark")

print(f"Model loading time       : {ml_time:.4f} seconds")
print(f"ML inference time       : {ml_inference_time:.4f} seconds")
print(f"Embedding generation   : {em_generation_time:.4f} seconds")
print(f"FAISS search time      : {faiss_time:.4f} seconds")
print(f"RAG retrieval time     : {rag_time:.4f} seconds")
print(f"LLM inference time     : {llm_inference_time:.4f} seconds")
print(f"End-to-end response    : {end_to_end_time:.4f} seconds")
print(f"RAM before             : {ram_before:.2f} MB")
print(f"RAM after              : {ram_after:.2f} MB")
print(f"Additional RAM         : {ram_after - ram_before:.2f} MB")

System Resource Benchmark
Model loading time       : 0.0111 seconds
ML inference time       : 0.0130 seconds
Embedding generation   : 0.0432 seconds
FAISS search time      : 0.0030 seconds
RAG retrieval time     : 0.0497 seconds
LLM inference time     : 9.1795 seconds
End-to-end response    : 11.2063 seconds
RAM before             : 381.15 MB
RAM after              : 370.20 MB
Additional RAM         : -10.95 MB


#### Resource Benchmarking: 
    The system showed low resource requirements for ML, embeddings, and FAISS. End-to-end response time was 11.2063 seconds, with local LLM inference as the main bottleneck at 9.1795 seconds. RAM usage remained around 370–381 MB, indicating that the platform is practical for CPU-based execution on a low-end laptop.
